# Chapter 4: Baker's Method & Continued Fraction Reduction (Dihedral Cases)

**Context:**
This script resolves the Dihedral cases ($\mathcal{G}_{t} \simeq D_4$) for the family of Thue equations $F_t(X,Y) = \mu$. In these cases, the roots $|\alpha|$ and $|\alpha'|$ are multiplicatively dependent. This dependence renders Laurent's Theorem on linear forms in two logarithms inapplicable, necessitating a specialized approach.

**Methodology:**
1. **Mignotte's Theorem:** Replaces Laurent's Theorem with a complex linear form in logarithms. The script then applies Mignottes theorem to establish an initial absolute upper bound of $\log|y| < 9.7 \times 10^5$.
2. **Dihedral Continued Fraction (CF) Reduction:** Calculates the multiplicative dependence integers $(a, b)$ for the roots $\alpha, \alpha'$. It then uses the continued fraction expansion of the irrational number $|\log(\alpha^a {\alpha'}^{-b})| / \pi$ to reduce the initial upper bound on $|y|$ to $< 40$.

**Key Functions:**
* `find_min_logalpha_D4()`: Bounds $|\log|\alpha||$, restricted to the $D_4$ cases.
* `test_mignotte_bounds()`: Computes the initial upper bounds on $\log|y|$ across combinations of $|t|$ and $\lambda$.
* `apply_Legendre_D4_reduction()`: Performs the continued fraction reduction and directly checks possible $(x,y)$ for any uneliminated $t$.

* *Note: 200-bit precision is used for the continued fraction reduction.*

In [ ]:
import itertools
import math
from sage.all import *

# =============================================================================
# GLOBAL SETTINGS & HIGH PRECISION FIELDS
# =============================================================================
RUN_EXHAUSTIVE_SEARCH = False

# 200-bit precision for CF reductions.
RR_val = RealField(200)
CC_val = ComplexField(200)

# =============================================================================
# SHARED HELPER FUNCTIONS & DATA GENERATION
# =============================================================================

def get_d4_quad_ints(M):
    """Returns valid imaginary quadratic integers for G = D4."""
    dlist = []
    for d in range(1, floor(4 * M**2 - 1) + 1):
        rem = d % 4
        if (rem in (1, 2) and d <= M**2) or rem == 3:
            if Integer(d).is_squarefree():
                dlist.append(d)
                
    big_set = {}
    for d in dlist:
        ablist = set()
        for i in range(1, ceil(M / sqrt(d))):
            ablist.add((0, i))
        if d == 14 and M >= sqrt(23):
            ablist.add((3, 1))
        big_set[d] = ablist
    return dlist, big_set

def generate_d4_t_data(m_bound):
    """Generator that yields the roots and absolute values for valid D4 t's."""
    dlist, all_ints = get_d4_quad_ints(m_bound)
    
    x_qq = PolynomialRing(QQ, 'x').gen()
    CC_poly = PolynomialRing(CC_val, 'x')
    x_cc = CC_poly.gen()
    
    for d in dlist:
        for a, b in all_ints[d]:
            abst = RR_val(sqrt(a**2 + d*b**2))
            
            dpol = x_qq**2 - 2 * a * x_qq + a**2 + d * b**2
            dpoldiscrim = Integer((2 * a)**2 - 4 * (a**2 + d * b**2))
            
            if dpoldiscrim.is_square():
                continue
                
            L = NumberField(dpol, 't')
            t_nf = L.gen()
            R = PolynomialRing(L, 'y')
            y = R.gen()
            
            f = y**2 - t_nf * y - 4
            ft = y**4 - t_nf * y**3 - 6 * y**2 + t_nf * y + 1
            
            if not f.is_irreducible() or not ft.is_irreducible():
                continue
                
            tt = CC_val(a + sqrt(-d)*b)
            ftt = x_cc**4 - tt*x_cc**3 - 6*x_cc**2 + tt*x_cc + 1
            ftp = ftt.derivative()
            
            roots = [CC_val(r[0]) for r in ftt.roots()]
            
            alpha0 = min(roots, key=abs)
            a3 = CC_val(-(alpha0 + 1)/(alpha0 - 1))
            
            yield d, a, b, abst, alpha0, a3, ftp, roots


# =============================================================================
# PART 1: Bounding |log|alpha|| for D4 Cases
# =============================================================================

def find_min_logalpha_D4(m_bound):
    minlogalpha = RR_val(10.0)
    for d, a, b, abst, alpha0, a3, ftp, roots in generate_d4_t_data(m_bound):
        logalpha = RR_val(abs(log(abs(alpha0))))
        if logalpha < minlogalpha:
            minlogalpha = logalpha
    return minlogalpha


# =============================================================================
# PART 2: Mignotte's Theorem Bounds on log|y|
# =============================================================================

def get_mignotte_constants_fast(lambdaval, abst):
    D = 4.0
    lambdaval = float(lambdaval)
    abst = float(abst)
    
    log_abst = math.log(abst)
    IndexBd = 16.0 * log_abst**2 + 24.16
    h = 1.5 * log_abst + 2.64
    
    rho = math.exp(lambdaval)
    a_val = 0.5 * rho * math.pi + D * h
    s1 = 2.0 * math.pi * rho
    s_val = 1.0 / (3.0 * s1) + 1.0 / (24.0 * s1 * (1.0 + s1 / (3.0 * lambdaval)))
    
    rootknum = 1.0/3.0 + math.sqrt(1.0/9.0 + 2.0 * math.pi * s_val)
    k_val = (rootknum / lambdaval)**2
    
    term1a = math.log(1.0 / (math.pi * rho) + 1.0 / (2.0 * a_val))
    term1_const = D * term1a - D * math.log(math.sqrt(k_val)) + 0.833
    term2a = 1.0 / (6.0 * rho * math.pi) + 1.0 / (3.0 * a_val)
    term2 = 3.0 * lambdaval / 2.0 + (1.0 / k_val) * term2a + 0.023
    
    term1_L_const1 = 8.0 * math.pi * k_val * rho * (1.0/lambdaval)
    term2_L_const = 0.5 * lambdaval + 2.0 * math.log(lambdaval) - (D + 2.0) * math.log(2.0)
    
    return D, IndexBd, a_val, term1_const, term2, term1_L_const1, term2_L_const

def mignotte_lambda_bound_fast(logy_guess, D, IndexBd, a_val, term1_const, term2, term1_L_const1, term2_L_const):
    B = IndexBd * (4.51 * logy_guess + 14.9)
    H_val = D * math.log(B) + term1_const + term2
    
    term1_L = term1_L_const1 * H_val**2 + 0.37
    term2_L = -2.0 * math.log(H_val) + term2_L_const
    
    LambdaLB = -a_val * term1_L + term2_L
    return LambdaLB

def find_mignotte_bound(lambdaval, abst, increment):
    Ct = 126.95
    abst = float(abst)
    log_abst = math.log(abst)
    Ival = 16.0 * log_abst**2 + 24.16
    log_Ct_Ival = math.log(5.0 * Ival * Ct)
    
    constants = get_mignotte_constants_fast(float(lambdaval), abst)
    
    logybd = 200000.0 / float(increment)
    L_val, R_val = 0.0, 1.0
    inc = float(increment)
    
    while L_val < R_val:
        logybd *= inc
        L_val = 4.0 * logybd + log_abst
        R_val = -mignotte_lambda_bound_fast(logybd, *constants) + log_Ct_Ival
        
    return logybd

def test_mignotte_bounds(lambdainc, tlb, tub, increment):
    intervalbd = 1.0
    
    norm_start = int(tlb**2)
    norm_end = int(tub**2) + 1
    
    lambda_start = int(1.8 / lambdainc)
    lambda_end = int(3.0 / lambdainc + 1.0 / lambdainc)
    lambda_vals = [float(idx * lambdainc) for idx in range(lambda_start, lambda_end)]
    
    for normt in range(norm_start, norm_end):
        abst = math.sqrt(normt)
        overallbd = 10**10
        
        for lambdaval in lambda_vals:
            bd = find_mignotte_bound(lambdaval, abst, float(increment))
            if bd < overallbd:
                overallbd = bd
                
        if overallbd > intervalbd:
            intervalbd = overallbd
            
    return intervalbd


# =============================================================================
# PART 3: Continued Fraction Reduction (Dihedral Specific)
# =============================================================================

def calculate_Bt_Ct(derivativeval, diff1, diff2, abst):
    Bt = 8 / (derivativeval / abst)
    z1num, z2num = Bt / diff1, Bt / diff2
    z1bd, z2bd = z1num / (40**4 * abst), z2num / (40**4 * abst)
    z1num2, z2num2 = z1num / (1 - z1bd), z2num / (1 - z2bd)
    Ct = z1num2 + z2num2
    return ceil(1000 * Bt) / 1000, ceil(1000 * Ct) / 1000

def Imax(abst):
    if abst < 10: return (4 / 0.253) * (log(abst)**2 + 1.51)
    return (4 / 0.253) * (log(abst + 0.802)**2 + log(1.20405)**2)

def new_maxdenom(log_yval, abst, j):
    base = (4 / 0.254) * (math.log(float(abst))**2 + 1.51)
    if j == 0: return base * (2 * (1.78 * float(log_yval) + 1.48) + 1)
    if j == 3: return base * (2 * (1.78 * float(log_yval) + 2.64))

def find_min_y(ymax, abst, absalpha, Bt):
    var('x')
    beta_fn = absalpha * x**4 - 40 * x**3 + Bt / abst
    beta_fn_min_x = float(120 / (4 * absalpha))
    return find_root(beta_fn, beta_fn_min_x, float(ymax))

def iterative_cf_test(cf, index_bound, maxdenom, diffs, cst, Ibd, denom_cst, abst):
    max_yval = 0 
    for i in range(1, index_bound):
        pq = cf.convergent(i)
        q = pq.denominator()
        if q > maxdenom: break
            
        RHS = float(diffs[i])
        if RHS == 0: continue
            
        LHS = float(cst * Ibd) / (float(denom_cst) * float(abst) * float(q))
        
        root = (LHS / RHS)**0.25
        yval = max(1, int(root) - 1)
        while float(LHS / (yval**4)) > RHS:
            yval += 1
            
        if yval > max_yval:
            max_yval = yval
            
    return max_yval

def get_fib_index(N):
    i = 0
    while fibonacci(i) < N: i += 1
    return i

def final_j3_elim(alpha3, alpha13diff, ymin, ymax, abst, dval, Bt):
    absalpha3 = abs(alpha3)
    x2set = set()
    Ltemp = QuadraticField(-dval, 'temproot')
    temproot = Ltemp.gen()
    
    for y2 in range(int(ymin**2), int(ymax**2) + 1):
        absy = sqrt(y2)
        error_term = RR_val(Bt / (absy**3 * abst))
        
        x2lb = max(ceil((absalpha3 * absy - error_term)**2), 35**2)
        x2ub = floor((absalpha3 * absy + error_term)**2)
        
        if x2lb <= x2ub:
            for x2 in range(x2lb, x2ub + 1):
                xy = QQ(x2 / y2)
                if xy > (alpha13diff - 1/absalpha3 - RR_val(Bt/(y2**2 * abst))):
                    continue
                if xy.is_norm(Ltemp):
                    for x_el, y_el in itertools.product(Ltemp.elements_of_norm(x2), Ltemp.elements_of_norm(y2)):
                        ratio = CC_val(x_el / y_el)
                        if abs(ratio - alpha3) < RR_val(Bt/absy) or abs(ratio + alpha3) < RR_val(Bt/absy):
                            x_tr, y_tr = x_el.trace() / 2, y_el.trace() / 2
                            x_im = ((x_el - x_tr) / temproot) * sqrt(-dval)
                            y_im = ((y_el - y_tr) / temproot) * sqrt(-dval)
                            x2set.add((x2, y2, x_tr + x_im, y_tr + y_im))
    return x2set

def apply_Legendre_D4_reduction(m_bound=100):
    for d, a, b, abst, alpha0, a3, ftp, roots in generate_d4_t_data(m_bound):
        alphaPrime = CC_val((alpha0 - 1) / (alpha0 + 1))
        
        ftpa0, ftpa3 = abs(ftp(alpha0)), abs(ftp(a3))
        a01diff, a03diff = abs(alpha0 + 1/a3), abs(alpha0 - a3)
        a23diff, a13diff = abs(a3 + CC_val(1/alpha0)), abs(a3 + 1/a3)
        
        Bt0, Ct0 = calculate_Bt_Ct(ftpa0, a01diff, a03diff, abst)
        Bt3, Ct3 = calculate_Bt_Ct(ftpa3, a23diff, a03diff, abst)
        Ibd = Imax(abst)
        
        aval, bval = 0, 1
        for coefs in [(0, 1), (1, 1), (-1, 1), (5, 1), (-1, 5), (-5, 1), (1, 5)]:
            val1 = abs(coefs[0]*log(abs(alpha0)) + coefs[1]*log(abs(alphaPrime)))
            val2 = abs(coefs[0]*log(abs(alpha0)) - coefs[1]*log(abs(alphaPrime)))
            if val1 < 1e-10 or val2 < 1e-10: 
                aval, bval = coefs[0], coefs[1]
                
        xi = alpha0**aval * alphaPrime**bval
        irrational = RR_val(abs(log(xi)) / pi)
        cf = continued_fraction(irrational)
        
        ymax_log = 7.1 * 10**5
        maxdenom0 = new_maxdenom(ymax_log, abst, 0)
        maxdenom3 = new_maxdenom(ymax_log, abst, 3)
        
        idx_bound = get_fib_index(max(maxdenom0, maxdenom3))
        diffs = [RR_val(abs(irrational - cf.convergent(i))) for i in range(idx_bound + 1)]
        
        ylb = find_min_y(ymax_log, abst, RR_val(abs(alpha0)), Bt0)
        ylb3 = 40
        
        yval0 = iterative_cf_test(cf, idx_bound, maxdenom0, diffs, Ct0, Ibd, math.pi, abst)
        yval3 = iterative_cf_test(cf, idx_bound, maxdenom3, diffs, Ct3, Ibd, math.pi, abst)
        
        maxdenom0 = new_maxdenom(math.log(max(yval0, 2)), abst, 0)
        maxdenom3 = new_maxdenom(math.log(max(yval3, 2)), abst, 3)
        idx_bound = get_fib_index(max(maxdenom0, maxdenom3))
        
        if not (yval0 < ylb and yval3 < ylb3):
            yval0 = iterative_cf_test(cf, idx_bound, maxdenom0, diffs, Ct0, Ibd, math.pi, abst)
            yval3 = iterative_cf_test(cf, idx_bound, maxdenom3, diffs, Ct3, Ibd, math.pi, abst)
            
        maxdenom0 = new_maxdenom(math.log(max(yval0, 2)), abst, 0)
        idx_bound = get_fib_index(maxdenom0)
        
        if not (yval0 < ylb and yval3 < ylb3):
            yval0 = iterative_cf_test(cf, idx_bound, maxdenom0, diffs, Ct0, Ibd, math.pi, abst)
            yval3 = iterative_cf_test(cf, idx_bound, maxdenom3, diffs, Ct3, Ibd, math.pi, abst)
            
        if yval0 >= ylb:
            print(f"t = {a}+sqrt({-d})*{b}, |y|<{yval0}, |y|>{ylb}, j=0")
        if yval3 >= ylb3:
            xyset = final_j3_elim(a3, a13diff, ylb3, yval3, abst, d, Bt3)
            if xyset:
                print(f"t = {a}+sqrt({-d})*{b}, j=3, Survivors: {xyset}")


# =============================================================================
# MAIN RUNNER
# =============================================================================

if __name__ == "__main__":
    if RUN_EXHAUSTIVE_SEARCH:
        print("=== Lemma 4.8: Bounding |log|alpha|| for D4 Cases ===")
        min_alpha_d4 = find_min_logalpha_D4(100)
        print(f"  • Min |log|alpha||: {min_alpha_d4:.4f}")
        
        print("\n=== Lemma 4.8: Mignotte's Theorem log|y| Bound ===")
        mignotte_bound = test_mignotte_bounds(lambdainc=0.1, tlb=1, tub=100, increment=1.01)
        print(f"  • log|y| < {mignotte_bound:.4f}")
        
        print("\n=== Lemma 4.8: Dihedral CF Reductions ===")
        print("Executing iterative reductions...")
        apply_Legendre_D4_reduction(m_bound=100)
        print("Reduction complete. If no solutions printed above, none exist > 40.")